## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# CNN applied to images - Application: posture classification

In 06.02 we saw how kernels change an image, trained a small CNN on handwritten digits, and reused it for clothes (transfer learning). Now we use CNNs for a movement-science task: classify images of people in three postures, **lying, sitting and standing**.

<div style="text-align:center">
<img src="images/classes.png" alt="plot" width="400" />
</div>

These images are much harder than MNIST: they are larger, in color, the people have different sizes, positions, clothes and backgrounds, and we have only about 1,000 of them.

We compare three approaches:
1. a small CNN trained **from scratch**,
2. a CNN **pretrained** on ImageNet (a large database of labeled photographs, explained below), used as a **feature extractor** (its embedding goes to a logistic regression),
3. the same pretrained CNN, **fine-tuned** with our images.

**Run time:** parts 1 and 2 take a few minutes on a CPU. Part 3 (fine-tuning) is much faster with a GPU: in Colab, use Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'   # for Mac computers (must be set before importing torch)
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

## The data

The images are organized in folders, one folder per class:

- `data/posture-classification/train/lying`, `.../sitting`, `.../standing` (964 images)
- `data/posture-classification/valid/lying`, `.../sitting`, `.../standing` (108 images)

With this structure, `datasets.ImageFolder` loads the images and uses the folder name as the label.

As in module 05, we need **three** sets:
- **train**: to train the model,
- **validation**: to choose the settings (for example the best epoch),
- **test**: to measure the final model **one time**.

We use the `valid` folder as the **test** set, and we keep 15% of the `train` folder as the **validation** set.

## Transformations

The images have different sizes, but a CNN processes images of the same size in a batch. `Resize((224, 224))` changes every image to 224 × 224 pixels. (It does not crop or pad, so the image can be stretched a little.) `ToTensor()` changes the image into a tensor with values from 0 to 1.

We can also add random transformations to the training images, called **augmentations**: for example, a random horizontal flip and a small random rotation. Each epoch, the model sees a slightly different version of each image, which makes overfitting less likely. Augmentations are applied **only to the training data**.

In [ ]:
train_dir = "data/posture-classification/train"
test_dir = "data/posture-classification/valid"

def make_loaders(train_transform, eval_transform, batch_size=32):
    """Build the three data loaders. Call it again every time you change the transforms."""
    full_train = datasets.ImageFolder(train_dir, transform=train_transform)
    full_train_eval = datasets.ImageFolder(train_dir, transform=eval_transform)   # same images, no augmentation
    test_set = datasets.ImageFolder(test_dir, transform=eval_transform)
    idx = np.random.RandomState(0).permutation(len(full_train))
    n_val = int(0.15 * len(idx))
    train_set, val_set = Subset(full_train, idx[n_val:]), Subset(full_train_eval, idx[:n_val])
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(val_set, batch_size=batch_size, shuffle=False),
            DataLoader(test_set, batch_size=batch_size, shuffle=False),
            full_train.classes)

train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                                      transforms.RandomRotation(10), transforms.ToTensor()])
eval_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
train_loader, val_loader, test_loader, class_names = make_loaders(train_transform, eval_transform)

y_test = np.array([label for _, label in test_loader.dataset.samples])
print("Classes:", class_names)
print("Images: train", len(train_loader.dataset), "| validation", len(val_loader.dataset), "| test", len(test_loader.dataset))
print("Test images per class:", dict(zip(class_names, np.bincount(y_test).tolist())))
print(f"Baseline (always the most common class): test accuracy = {np.bincount(y_test).max() / len(y_test):.3f}")

In [ ]:
images, labels = next(iter(train_loader))
plt.figure(figsize=(14, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0).numpy())      # (channels, H, W) -> (H, W, channels) for plotting
    plt.title(class_names[labels[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

## Helper functions

The training and evaluation functions are the same as in module 05: train with batches, compute the validation loss after each epoch, and keep the best epoch (**early stopping**).

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, preds, ys = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total_loss += criterion(out, y).item() * len(y)
        preds.append(out.argmax(1).cpu()); ys.append(y.cpu())
    preds, ys = torch.cat(preds).numpy(), torch.cat(ys).numpy()
    return total_loss / len(ys), (preds == ys).mean(), preds

def fit(model, optimizer, epochs):
    criterion = nn.CrossEntropyLoss()
    best_loss, best_state = np.inf, None
    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, _ = evaluate(model, val_loader, criterion)
        if va_loss < best_loss:
            best_loss, best_state = va_loss, copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch+1:02d} | train loss {tr_loss:.3f}, acc {tr_acc:.3f} | validation loss {va_loss:.3f}, acc {va_acc:.3f}")
    model.load_state_dict(best_state)
    return model

def test_model(model, title):
    _, acc, pred = evaluate(model, test_loader, nn.CrossEntropyLoss())
    print(f'{title}: test accuracy = {acc:.3f}')
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=class_names, cmap='Blues')
    plt.title(title); plt.show()
    return acc

## 1. A small CNN trained from scratch

Three convolution blocks (convolution, batch normalization, ReLU, max pooling), then **global average pooling**: the mean of each of the 64 feature maps. So the embedding has 64 numbers, and the classifier is small.

(If we flattened the last feature maps instead, the embedding would have 64 × 28 × 28 = 50,176 numbers, and the first fully connected layer alone would have more than 12 million parameters: far too many for 800 training images.)

In [ ]:
class SimplePostureCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),          # global average pooling: (64, 28, 28) -> (64, 1, 1)
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(0)
small_cnn = SimplePostureCNN(len(class_names)).to(device)
print(f'Parameters: {sum(p.numel() for p in small_cnn.parameters()):,}')

Before training, we can pass one image through the layers and look at some feature maps (the kernels are still random).

In [ ]:
small_cnn.eval()
x = images[0:1].to(device)     # one image: [1, 3, 224, 224]
with torch.no_grad():
    z = x
    for i, layer in enumerate(small_cnn.features):
        z = layer(z)
        if isinstance(layer, nn.MaxPool2d):
            feat = z.squeeze(0).cpu()
            plt.figure(figsize=(14, 2.5))
            for j in range(6):
                plt.subplot(1, 6, j + 1); plt.imshow(feat[j].numpy(), cmap='gray'); plt.axis('off')
                if j == 0: plt.title(f'After layer {i}: {tuple(feat.shape)}', fontsize=9, loc='left')
            plt.show()

Train for 10 epochs (a few minutes on a CPU).

In [ ]:
optimizer = torch.optim.Adam(small_cnn.parameters(), lr=1e-3)
small_cnn = fit(small_cnn, optimizer, epochs=10)
acc_small = test_model(small_cnn, 'Small CNN from scratch')

The small CNN is only a little better than the baseline. On MNIST (06.02), a small CNN from scratch worked well, because the digits are simple, centered and on a black background. Here the images are much more varied, and with about 800 training images a CNN cannot learn good features from scratch.

The usual solution is to start from a model that other people trained on a very large dataset. Such models are sometimes called **foundation models** or **pretrained models**. We use **ResNet18**, a CNN pretrained on **ImageNet**. This is transfer learning as in 06.02, but now from a very large and varied dataset.

**What is ImageNet?** ImageNet is a large public database of photographs, built to train and compare image-recognition models. The version used to train models has about **1.2 million training images** in **1,000 categories** (animals, objects, food, vehicles, scenes, etc.), and each image was labeled by people with its main object. A model trained on ImageNet learns general visual features (edges, textures, shapes, parts of objects) that are useful for many other image tasks. Lecture 07.01 gives more details on ImageNet and on other datasets used to train and compare models.

**Important:** a pretrained model expects the same input scaling as during its training. For ImageNet models, each color channel is normalized with the ImageNet mean and standard deviation. After we change the transforms, we must **build the datasets and loaders again**, or the new transforms are not used.

In [ ]:
imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                                      transforms.RandomRotation(10), transforms.ToTensor(), imagenet_norm])
eval_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), imagenet_norm])
train_loader, val_loader, test_loader, class_names = make_loaders(train_transform, eval_transform)   # rebuild!

## 2. Pretrained CNN as a feature extractor

We remove the last layer of ResNet18 (`fc`), so the network gives the **512-number embedding** of each image. We do not train the CNN at all. Then a logistic regression (module 04) uses the embeddings to classify the posture.

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = nn.Identity()            # remove the classification layer: output = 512-number embedding
resnet = resnet.to(device).eval()

@torch.no_grad()
def embed(loader):
    E, Y = [], []
    for x, y in loader:
        E.append(resnet(x.to(device)).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(E), np.concatenate(Y)

emb_train_loader, emb_val_loader, emb_test_loader, _ = make_loaders(eval_transform, eval_transform)   # no augmentation for the embeddings
E_train, Y_train = embed(emb_train_loader)
E_val, Y_val = embed(emb_val_loader)
E_test, Y_test = embed(emb_test_loader)
print('Embeddings:', E_train.shape, E_test.shape)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(C=0.1, max_iter=3000))])
lr.fit(np.vstack([E_train, E_val]), np.concatenate([Y_train, Y_val]))     # no epochs to choose, so we can use train + validation
pred = lr.predict(E_test)
acc_features = accuracy_score(Y_test, pred)
print(f'Pretrained ResNet18 features + logistic regression: test accuracy = {acc_features:.3f}')
ConfusionMatrixDisplay.from_predictions(Y_test, pred, display_labels=class_names, cmap='Blues')
plt.title('ResNet18 features + logistic regression'); plt.show()

Without any training of the CNN, the pretrained features are much better than the small CNN. The features learned on ImageNet (edges, textures, shapes, body parts) are useful for our task too. This is called **transfer learning**.

## 3. Fine-tuning

Now we train the **whole** pretrained network with our images, with a new last layer for our 3 classes. We use a small learning rate, because the pretrained weights are already good and we only want to adjust them.

This part takes about 10 minutes on a CPU, and less than 1 minute on a GPU.

In [ ]:
torch.manual_seed(0)
finetuned = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
finetuned.fc = nn.Linear(finetuned.fc.in_features, len(class_names))     # new last layer for 3 classes
finetuned = finetuned.to(device)
optimizer = torch.optim.Adam(finetuned.parameters(), lr=1e-4)
finetuned = fit(finetuned, optimizer, epochs=5)
acc_finetuned = test_model(finetuned, 'Fine-tuned ResNet18')

In [ ]:
print(f"Baseline:                         {np.bincount(y_test).max() / len(y_test):.3f}")
print(f"Small CNN from scratch:           {acc_small:.3f}")
print(f"ResNet18 features + log. regr.:   {acc_features:.3f}")
print(f"Fine-tuned ResNet18:              {acc_finetuned:.3f}")

Look at the fine-tuning output: the validation accuracy (about 0.94) is higher than the test accuracy. The validation images come from the same folder as the training images, so they are more similar to them; the test images come from another folder. This is why we keep a separate test set.

Remember also that the test set has only 108 images: a difference of 1 image is about 1% of accuracy. Also, 4 of the test images come from the same source images as some training images, which makes the test a little optimistic.

## What does the pretrained network see?

We pass one image through the main blocks of ResNet18 and show the mean of the feature maps of each block. The maps become smaller (lower resolution) and more abstract in the deeper blocks.

In [ ]:
finetuned.eval()
x, label = test_loader.dataset[0]
x = x.unsqueeze(0).to(device)
mean, std = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])
stage_outputs = [("input", (x[0].cpu() * std[:, None, None] + mean[:, None, None]).permute(1, 2, 0).clamp(0, 1))]
with torch.no_grad():
    z = x
    for name_, layer in [("conv1", finetuned.conv1), ("bn1", finetuned.bn1), ("relu", finetuned.relu),
                         ("maxpool", finetuned.maxpool), ("layer1", finetuned.layer1), ("layer2", finetuned.layer2),
                         ("layer3", finetuned.layer3), ("layer4", finetuned.layer4)]:
        z = layer(z)
        stage_outputs.append((name_, z.cpu()))

fig, axes = plt.subplots(1, len(stage_outputs), figsize=(3 * len(stage_outputs), 3))
for ax, (name_, feat) in zip(axes, stage_outputs):
    if name_ == "input":
        ax.imshow(feat.numpy()); ax.set_title(f"Input ({class_names[label]})", fontsize=9)
    else:
        ax.imshow(feat[0].mean(dim=0).numpy(), cmap="magma"); ax.set_title(f"{name_}\n{tuple(feat.shape[1:])}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()